# Data cleaning

Data loading and feature engineering

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = pd.read_csv("../data/raw/produkcja.csv")

power = (df['Rotational speed [rpm]'] * df['Torque [Nm]'] * (2 * np.pi / 60)).round(2)
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


Adding columns: Temperature difference and Power [W]

In [2]:
df.insert(loc=5, column='Temperature difference', value= df['Process temperature [K]']- df['Air temperature [K]'] )
df.insert(loc=8, column = 'Power [W]', value = power)

In [3]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,10.5,1551,42.8,6951.59,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,10.5,1408,46.3,6826.72,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,10.4,1498,49.4,7749.39,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,10.4,1433,39.5,5927.50,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,10.5,1408,40.0,5897.82,9,0,0,0,0,0,0


Finding Machine failure records with no failure subtype assigned

In [4]:
df[((df['TWF'] == 0) & (df['HDF'] == 0) & (df['PWF'] == 0) & (df['OSF'] == 0) & (df['RNF'] == 0)) & (df['Machine failure'] == 1)]

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
1437,1438,H30851,H,298.8,309.9,11.1,1439,45.2,6811.27,40,1,0,0,0,0,0
2749,2750,M17609,M,299.7,309.2,9.5,1685,28.9,5099.49,179,1,0,0,0,0,0
4044,4045,M18904,M,301.9,310.9,9.0,1419,47.7,7088.09,20,1,0,0,0,0,0
4684,4685,M19544,M,303.6,311.8,8.2,1421,44.8,6666.54,101,1,0,0,0,0,0
5536,5537,M20396,M,302.3,311.8,9.5,1363,54.0,7707.58,119,1,0,0,0,0,0
5941,5942,L53121,L,300.6,310.7,10.1,1438,48.5,7303.47,78,1,0,0,0,0,0
6478,6479,L53658,L,300.5,309.8,9.3,1663,29.1,5067.73,145,1,0,0,0,0,0
8506,8507,L55686,L,298.4,309.6,11.2,1710,27.3,4888.63,163,1,0,0,0,0,0
9015,9016,L56195,L,297.2,308.1,10.9,1431,49.7,7447.74,210,1,0,0,0,0,0


Editing these records: in the EDA their parameters do not deviate from the norm – treated as label noise.

In [5]:
df.loc[df[((df['TWF'] == 0) & (df['HDF'] == 0) & (df['PWF'] == 0) & (df['OSF'] == 0) & (df['RNF'] == 0)) & (df['Machine failure'] == 1)].index, 'Machine failure'] = 0

Confirming the records were actually changed

In [6]:
df[((df['TWF'] == 0) & (df['HDF'] == 0) & (df['PWF'] == 0) & (df['OSF'] == 0) & (df['RNF'] == 0)) & (df['Machine failure'] == 1)].shape[0]

0

In [7]:
df[['Air temperature [K]', 'Process temperature [K]', 'Temperature difference']].corr()

,Air temperature [K],Process temperature [K],Temperature difference
Air temperature [K],1.000000,0.876107,-0.699583
Process temperature [K],0.876107,1.000000,-0.268413
Temperature difference,-0.699583,-0.268413,1.000000


Dropping columns that could cause data leakage, plus columns that add nothing for the model

In [8]:
df.drop(columns=['UDI', 'Product ID', 'Air temperature [K]', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'], inplace=True)

In [9]:
df.head()

,Type,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min],Machine failure
0,M,308.6,10.5,1551,42.8,6951.59,0,0
1,L,308.7,10.5,1408,46.3,6826.72,3,0
2,L,308.5,10.4,1498,49.4,7749.39,5,0
3,L,308.6,10.4,1433,39.5,5927.50,7,0
4,L,308.7,10.5,1408,40.0,5897.82,9,0


One-hot encoding of the machine type

In [10]:
dummies = pd.get_dummies(df, columns=['Type'], drop_first=True)

df =df.drop(columns=['Type'])
df.insert(loc=0, column='Type_L', value=dummies['Type_L'])
df.insert(loc=1, column='Type_M', value=dummies['Type_M'])

